# Recombination landscape and CO/NCO

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as patches
from matplotlib import colormaps
from matplotlib.patches import PathPatch
from matplotlib.path import Path as MplPath
from matplotlib.lines import Line2D
import matplotlib.cm
from pathlib import Path
import tqdm
import sys
import seaborn as sns
import scipy.stats
import msprime
import os
import joblib
import polars as pl
import glob
import logging

os.environ["PATH"] += ":" + os.path.join(sys.prefix, "bin")
os.environ["PATH"] += ":" + "/software/treeoflife/shpc/0.1.26/wrapper/quay.io/biocontainers/bedtools/2.31.1--hf5e1c6e_1/bin"
import pybedtools

pd.set_option('display.max_rows', 1000)
pl.Config.set_tbl_rows(-1)
pl.Config.set_fmt_str_lengths(50)
plt.rcParams["pdf.use14corefonts"] = True
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

In [ ]:
repo = Path(os.getcwd())
if repo.name == "notebooks":
    repo = repo.parent
elif not (repo / "configs").exists():
    repo = Path("/nfs/users/nfs_r/rs42/rs42/git/recombination")

sys.path.append(str(repo))
from src.IDs import *
from src import liftover

aut_chrom_names = [f"chr{i}" for i in list(range(1, 23))]

grch37_chromosome_sizes_in_bp = {
    'chr1': 249250621,
    'chr2': 243199373,
    'chr3': 198022430,
    'chr4': 191154276,
    'chr5': 180915260,
    'chr6': 171115067,
    'chr7': 159138663,
    'chr8': 146364022,
    'chr9': 141213431,
    'chr10': 135534747,
    'chr11': 135006516,
    'chr12': 133851895,
    'chr13': 115169878,
    'chr14': 107349540,
    'chr15': 102531392,
    'chr16': 90354753,
    'chr17': 81195210,
    'chr18': 78077248,
    'chr19': 59128983,
    'chr20': 63025520,
    'chr21': 48129895,
    'chr22': 51304566,
}

grch38_chromosome_sizes_in_bp = {
    'chr1': 248_956_422,
    'chr2': 242_193_529,
    'chr3': 198_295_559,
    'chr4': 190_214_555,
    'chr5': 181_538_259,
    'chr6': 170_805_979,
    'chr7': 159_345_973,
    'chr8': 145_138_636,
    'chr9': 138_394_717,
    'chr10': 133_797_422,
    'chr11': 135_086_622,
    'chr12': 133_275_309,
    'chr13': 114_364_328,
    'chr14': 107_043_718,
    'chr15': 101_991_189,
    'chr16': 90_338_345,
    'chr17': 83_257_441,
    'chr18': 80_373_285,
    'chr19': 58_617_616,
    'chr20': 64_444_167,
    'chr21': 46_709_983,
    'chr22': 50_818_468,
    'chrX': 156_040_895,
    'chrY': 57_227_415,
}

rate_maps = {}
for chrom in aut_chrom_names:
    rate_maps[chrom] = msprime.RateMap.read_hapmap(
        open(f"/lustre/scratch122/tol/projects/sperm/data/references/04.genetic_maps/01.Bherer_etal_SexualDimorphismRecombination/Refined_EUR_genetic_map_b37/male_{chrom}.txt"),
        sequence_length=grch37_chromosome_sizes_in_bp[chrom],
    )

figure_dir = repo / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

CO_color = globals().get("CO_color", "#4C78A8")
NCO_color = globals().get("NCO_color", "#54A24B")
background_color = globals().get("background_color", "#777777")

events_xlsx = Path("/lustre/scratch122/tol/projects/sperm/results/recombination_events_sperm_20250417.xlsx")
events_parquet = Path("/lustre/scratch122/tol/projects/sperm/results/recombination_events_sperm_20250325.parquet")
accepted_events_df = pl.read_excel(events_xlsx)
events_df = pl.read_parquet(events_parquet)

In [ ]:
rahbari_df = pl.read_csv(repo / "configs/Rahbari.tsv", separator="\t")

sudmant_df = (
    pl.read_csv(repo / "configs/Sudmant.tsv", separator="\t")
    .with_columns(
        pl.col("sample_set").cast(pl.String),
        pl.col("sample_id").cast(pl.String),
        pl.col("flow_cell").cast(pl.String),
    )
)

CEPH_df = pl.read_csv(repo / "configs/CEPH.tsv", separator="\t")
ceph_df = CEPH_df

ceph_sample_ids_no_gps = sorted([
    x for x in CEPH_df["sample_id"].unique().to_list()
    if x not in ["NA12889", "NA12890", "NA12891", "NA12892"]
])

sample_ids = rahbari_sample_ids + sudmant_sample_ids

## Events

In [ ]:
%%time
reads_filenames = (
    [
        (
            f"/lustre/scratch122/tol/projects/sperm/results/Rahbari_20250212/read_analysis/{sample_set}/{sample_id}/reads/{chrom}/all_reads_structure_annotated.parquet"
        )
        for sample_id, sample_set in tqdm.tqdm(rahbari_df.select("sample_id", "sample_set").unique().iter_rows())
        for chrom in aut_chrom_names
    ] +
    [
        (
            f"/lustre/scratch122/tol/projects/sperm/results/Sudmant_20241121/read_analysis/{sample_set}/{sample_id}/reads/{chrom}/all_reads_structure_annotated.parquet"
        )
        for sample_id, sample_set in tqdm.tqdm(sudmant_df.select("sample_id", "sample_set").unique().iter_rows())
        for chrom in aut_chrom_names
    ]
)

In [ ]:
%%time
def read_CO_NCO(filename):
    return (
        pl.scan_parquet(filename)
        .select(
            'read_name',
            'read_length',
            'chrom',
            'sample_id',
            'high_quality_snp_positions',
            'grch37_reference_start',
            'grch38_reference_start',
            'grch37_reference_end',
            'grch38_reference_end',
            'grch37_reference_start_cM',
            'grch37_reference_end_cM',
            'CO_active_interval_crossover_prob',
            'full_read_crossover_prob',
            'is_high_quality_read',
            'high_quality_classification_class',
            'high_quality_classification_in_detectable_class',
            'snp_positions_on_read',
            'idx_transitions',
            'is_contamination',
        )
        .filter("is_high_quality_read")
        .filter(~pl.col("is_contamination"))
        .filter(pl.col("high_quality_classification_class").is_in(["CO", "GC"]))
        .filter(pl.col("high_quality_snp_positions").list.len() >= 3)
        .filter(pl.col("CO_active_interval_crossover_prob") > 0)
        .collect(streaming=True)
    )

CO_NCO_df = pl.concat(
    joblib.Parallel(n_jobs=-1, verbose=1)(
        joblib.delayed(read_CO_NCO)(filename) for filename in reads_filenames
    )
)

CO_NCO_df = (CO_NCO_df
    .with_columns(
        grch37_recombining_interval_start_pos = pl.col("grch37_reference_start") + pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(0)),
        grch37_recombining_interval_end_pos = pl.col("grch37_reference_start") + pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(-1) + 1),
        grch38_recombining_interval_start_pos = pl.col("grch38_reference_start") + pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(0)),
        grch38_recombining_interval_end_pos = pl.col("grch38_reference_start") + pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(-1) + 1),
        grch37_first_converted_marker_pos = pl.col("grch37_reference_start") + pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(0)+1),
    )
    .with_columns(
        grch37_recombining_interval_length = pl.col("grch37_recombining_interval_end_pos") - pl.col("grch37_recombining_interval_start_pos"),
        grch38_recombining_interval_length = pl.col("grch38_recombining_interval_end_pos") - pl.col("grch38_recombining_interval_start_pos"),
    )
)

dfs = []
for [chrom], df in CO_NCO_df.partition_by(by=["chrom"], as_dict=True).items():
    rate_map = rate_maps[chrom]
    dfs.append(
        df.with_columns(
            grch37_recombining_interval_start_poses_cm = rate_map.get_cumulative_mass(df["grch37_recombining_interval_start_pos"]) * 1e2,
            grch37_recombining_interval_end_poses_cm = rate_map.get_cumulative_mass(df["grch37_recombining_interval_end_pos"]) * 1e2,
        ).with_columns(
            grch37_recombining_interval_cM = (pl.col("grch37_recombining_interval_end_poses_cm") - pl.col("grch37_recombining_interval_start_poses_cm")),
            grch37_cM_per_bp_across_recombining_interval = (pl.col("grch37_recombining_interval_end_poses_cm") - pl.col("grch37_recombining_interval_start_poses_cm")) / pl.col("grch37_recombining_interval_length"),
            grch37_first_converted_marker_poses_rate = rate_map.get_rate(df["grch37_first_converted_marker_pos"]) * 1e2,
        )
    )

CO_NCO_df = (pl.concat(dfs)
    .with_columns(
        (pl.col("full_read_crossover_prob") * 1e2).alias("genetic_length_in_cm"),
        (pl.col("read_length")).alias("genetic_length_in_bp"),
        (pl.col("full_read_crossover_prob") * 1e2 / (pl.col("read_length")*1e-6)).alias("read_recomb_rate_in_cm_bp"),
    )
)

In [ ]:
%%time
def read_background_sample(filename):
    df = (
        pl.scan_parquet(filename)
        .select(
            'read_name',
            'chrom',
            'read_length',
            'grch38_reference_start',
            'grch38_reference_end',
            'CO_active_interval_crossover_prob',
            'full_read_crossover_prob',
            'is_high_quality_read',
            'is_contamination',
        )
        .filter("is_high_quality_read")
        .filter(~pl.col("is_contamination"))
        .filter(pl.col("CO_active_interval_crossover_prob") > 0)
        .collect(streaming=True)
    )
    return df.sample(fraction=1e-4, seed=1)

def read_background_rate(filename):
    return (
        pl.scan_parquet(filename)
        .select(
            'read_length',
            'CO_active_interval_crossover_prob',
            'full_read_crossover_prob',
            'is_high_quality_read',
            'is_contamination',
        )
        .filter("is_high_quality_read")
        .filter(~pl.col("is_contamination"))
        .filter(pl.col("CO_active_interval_crossover_prob") > 0)
        .select(
            n=pl.len(),
            rate_sum=(pl.col("full_read_crossover_prob") * 1e2 / (pl.col("read_length") * 1e-6)).sum(),
        )
        .collect(streaming=True)
    )

background_sample_df = pl.concat(
    joblib.Parallel(n_jobs=-1, verbose=1)(
        joblib.delayed(read_background_sample)(filename) for filename in reads_filenames
    )
)

background_rate_df = pl.concat(
    joblib.Parallel(n_jobs=-1, verbose=1)(
        joblib.delayed(read_background_rate)(filename) for filename in reads_filenames
    )
)

background_whole_read_rate = background_rate_df["rate_sum"].sum() / background_rate_df["n"].sum()

In [ ]:
%%time
def read_complex(filename):
    return (
        pl.scan_parquet(filename)
        .select(
            'read_length',
            'CO_active_interval_crossover_prob',
            'full_read_crossover_prob',
            'is_high_quality_read',
            'high_quality_classification_class',
            'is_contamination',
        )
        .filter("is_high_quality_read")
        .filter(~pl.col("is_contamination"))
        .filter(pl.col("high_quality_classification_class") == "CNCO")
        .filter(pl.col("CO_active_interval_crossover_prob") > 0)
        .collect(streaming=True)
    )

complex_filtered_df = pl.concat(
    joblib.Parallel(n_jobs=-1, verbose=1)(
        joblib.delayed(read_complex)(filename) for filename in reads_filenames
    )
)

## Genome-wide distribution

In [ ]:
cyto_T2T_df = pl.read_csv(
    "http://t2t.gi.ucsc.edu/chm13/hub/t2t-chm13-v2.0/download/chm13v2.0_cytobands_allchrs.bed.gz",
    separator="\t",
    new_columns=["chrom", "cyto_start", "cyto_end", "cyto_name", "type"],
    has_header=False,
)

cent_df = cyto_T2T_df.filter(pl.col("type") == "acen")

In [ ]:
trf_columns = ["start_pos_1based", "end_pos_1based", "repeat_length", "n_copies", "concensus_length", "percent_matches", "percent_indels", "alignment_score", "percent_A", "percent_C", "percent_G", "percent_T", "entropy", "concensus", "full_repeat", "flank_seq1", "flank_seq2"]

def open_trf(filename, chrom, start):
    return (
        pl.scan_csv(
            filename,
            has_header = False,
            separator = " ",
            comment_prefix = "@",
            new_columns = trf_columns,
        )
        .select(
            pl.lit(chrom).alias("chrom"),
            (pl.col("start_pos_1based") - 1 + int(start)).alias("start_pos_0based"),
            (pl.col("end_pos_1based") + int(start)).alias("end_pos_0based")
        )
    )

def open_all_trfs(chrom):
    dfs = [
        open_trf(f"/lustre/scratch122/tol/projects/sperm/results/Rahbari_20250212/global/t2t/{chrom}.{start}.fasta.trf.dat", chrom, start)
        for start in range(0, liftover.T2T_chromosome_sizes_in_bp[chrom], 10_000_000)
    ]
    return pl.concat(dfs).collect()

chrom_to_trf = {chrom: open_all_trfs(chrom) for chrom in tqdm.tqdm(aut_chrom_names)}

In [ ]:
chrom_to_mask_df = {}
resolution = 1_000_000

for chrom in tqdm.tqdm(aut_chrom_names):
    trf_bed = pybedtools.BedTool.from_dataframe(
        chrom_to_trf[chrom]
        .select("chrom", "start_pos_0based", "end_pos_0based")
        .to_pandas()
    )

    sdust = pybedtools.BedTool(f"/lustre/scratch122/tol/projects/sperm/results/Rahbari_20250212/global/T2T/{chrom}.fasta.sdust.dat")

    windows = pybedtools.BedTool([
        (chrom, x, x+resolution)
        for x in range(0, grch38_chromosome_sizes_in_bp[chrom] + resolution, resolution)
    ])

    masked = (sdust
        .cat(trf_bed, postmerge=False).sort()
        .merge()
    )

    chrom_to_mask_df[chrom] = pl.from_pandas(
        windows.intersect(masked, wao=True)
        .groupby(g=[1,2,3], c=7)
        .to_dataframe()
    )

In [ ]:
fig, axes = plt.subplots(
    ncols=2,
    nrows=22,
    width_ratios = (1, 60),
    figsize=(30, 20),
    facecolor="white",
)

plt.subplots_adjust(wspace=0, hspace=0)

from matplotlib import colors

max_color = 0.7
cmap = colormaps["Greys"]
new_cmap = mpl.colors.LinearSegmentedColormap.from_list(
    'Greys_half',
    cmap(np.linspace(0, max_color, 256))
)

norm = colors.TwoSlopeNorm(vcenter=0.2, vmin=0, vmax=1)

max_chrom_len = liftover.T2T_chromosome_sizes_in_bp[aut_chrom_names[0]]
for n_chrom, chrom in enumerate(aut_chrom_names):
    ax = axes[n_chrom, 0]
    ax.text(0.5, 0.5, chrom, horizontalalignment='center', verticalalignment='center', fontsize=24)
    ax.axis('off')

    ax = axes[n_chrom, 1]
    chr_len = liftover.T2T_chromosome_sizes_in_bp[chrom] / max_chrom_len

    height = 7.0
    height_ylim_pad = 1.0
    lower_anchor = 0.0
    ymid = height / 2
    curve = 0.01
    lw = 0.5
    edge_lw = 0.6
    event_height = 3
    event_alpha = 0.9

    cov_df = chrom_to_mask_df[chrom]
    for _, start, end, n_bp in cov_df.rows():
        color = new_cmap(norm(np.clip(n_bp / resolution, 0, max_color)))
        ax.add_patch(
            patches.Rectangle(
                xy = (start / max_chrom_len, 0),
                width = (end-start) / max_chrom_len,
                height = height,
                linewidth=0,
                facecolor=color,
            )
        )

    cdf = cent_df.filter(pl.col("chrom") == chrom)
    for _, start, end, _, _ in cdf.rows():
        ax.add_patch(
            patches.Rectangle(
                xy = (start / max_chrom_len, 0),
                width = (end-start) / max_chrom_len,
                height = height,
                linewidth=0,
                facecolor="red",
                alpha=0.2,
            )
        )

    outline = [
        (MplPath.MOVETO, (lower_anchor, height)),
        (MplPath.LINETO, (chr_len, height)),
        (MplPath.CURVE3, (chr_len  + curve, ymid)),
        (MplPath.LINETO, (chr_len, lower_anchor)),
        (MplPath.LINETO, (lower_anchor, lower_anchor)),
        (MplPath.CURVE3, (lower_anchor -  curve, ymid)),
        (MplPath.LINETO, (lower_anchor, height)),
        (MplPath.CLOSEPOLY, (lower_anchor, height)),
    ]

    codes, verts = zip(*outline)
    path = MplPath(verts, codes)
    patch = PathPatch(path, facecolor="none", edgecolor="black", linewidth=edge_lw)

    ax.add_patch(patch)
    ax.set_xlim(-0.1, 1+0.1)
    ax.set_ylim(-height_ylim_pad, height+height_ylim_pad)
    ax.axis('off')

    poses = events_df.filter((pl.col("event_type") == "CO") & (pl.col("chrom") == chrom))["T2T_reference_start"].sort().to_numpy()
    relative_poses = poses / max_chrom_len
    for pos in relative_poses:
        ax.plot([pos, pos], [height, height - event_height], color=CO_color, lw=lw, label="CO", alpha=event_alpha)

    poses = events_df.filter((pl.col("event_type") == "NCO") & (pl.col("chrom") == chrom))["T2T_reference_start"].sort().to_numpy()
    relative_poses = poses / max_chrom_len
    for pos in relative_poses:
        ax.plot([pos, pos], [lower_anchor, event_height], color=NCO_color, lw=lw, label="NCO", alpha=event_alpha)

fig.savefig(figure_dir / "all_chroms.pdf")

In [ ]:
fig, ax = plt.subplots(figsize=(1, 5))
sm = matplotlib.cm.ScalarMappable(norm=norm, cmap=new_cmap)
fig.colorbar(sm, cax=ax)
plt.show()

fig.savefig(figure_dir / "all_chroms_colorbar.pdf")

legend_elements = [
    Line2D([0], [0], color=CO_color, lw=3, label='CO'),
    Line2D([0], [0], color=NCO_color, lw=3, label='NCO'),
]

fig, ax = plt.subplots(figsize=(2.5, 2))
ax.axis('off')
legend = ax.legend(handles=legend_elements, loc='center', frameon=True, fontsize='large')
plt.show()

fig.savefig(figure_dir / "all_chroms_legend.pdf")

## Genetic length

In [ ]:
%%time
df = (CO_NCO_df
    .filter(pl.col("high_quality_snp_positions").list.len() >= 5)
    .filter(pl.col("high_quality_classification_in_detectable_class").is_not_null())
    .filter(pl.col("CO_active_interval_crossover_prob") > 0)
    .select(
        "CO_active_interval_crossover_prob",
        "high_quality_classification_in_detectable_class",
    )
)
CO_lens = df.filter(pl.col("high_quality_classification_in_detectable_class") == "CO")["CO_active_interval_crossover_prob"] * 1e2
NCO_lens = df.filter(pl.col("high_quality_classification_in_detectable_class") == "NCO")["CO_active_interval_crossover_prob"] * 1e2
all_lens = background_sample_df["CO_active_interval_crossover_prob"] * 1e2

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))

sns.histplot(
    [
        pd.Series(np.log10(CO_lens), name="CO"),
        pd.Series(np.log10(NCO_lens), name="NCO"),
        pd.Series(np.log10(all_lens), name="All reads"),
    ],
    bins=np.linspace(-5.5, 0.2, 50),
    stat="proportion",
    common_norm=False,
    element="step",
    palette=[CO_color, NCO_color, background_color],
    fill=False,
    alpha=0.9,
    ax=ax,
    lw=1,
)

ax.set_xlabel("Genetic length (cM)");
ax.set_xticks(
    np.arange(-5, 1),
    [f"$10^{{{x}}}$" for x in np.arange(-5, 1)],
);
ax.legend(
    ax.get_legend().legend_handles,
    ["CO reads", "NCO reads", "All reads"],
    loc="upper left",
);

ax.spines[['right', 'top']].set_visible(False)

fig.savefig(figure_dir / "sperm_co_nco_genetic_lengths.pdf")

## Blood genetic length

In [ ]:
%%time
ceph_good_samples = sorted(list(ceph_df["sample_id"].unique()))

blood_filenames = [
    f"/lustre/scratch122/tol/projects/sperm/results/CEPH_20250212/read_analysis/{sample_set}/{sample_id}/reads/{chrom}/all_reads_structure_annotated.parquet"
    for sample_id, sample_set in tqdm.tqdm(ceph_df.select("sample_id", "sample_set").unique().iter_rows())
    if sample_id in ceph_good_samples
    for chrom in aut_chrom_names
]

def read_blood_lens(filename):
    df = (
        pl.scan_parquet(filename)
        .select(
            'high_quality_snp_positions',
            'CO_active_interval_crossover_prob',
            'is_high_quality_read',
            "high_quality_classification_in_detectable_class",
        )
        .filter(pl.col("high_quality_snp_positions").list.len() >= 3)
        .filter("is_high_quality_read")
        .filter(pl.col("CO_active_interval_crossover_prob") > 0)
        .collect(streaming=True)
    )
    nco = (df
        .filter(pl.col("high_quality_snp_positions").list.len() >= 5)
        .filter(pl.col("high_quality_classification_in_detectable_class") == "NCO")
        .select("CO_active_interval_crossover_prob")
    )
    background = df.sample(fraction=1e-4, seed=1).select("CO_active_interval_crossover_prob")
    return nco, background

blood_parts = joblib.Parallel(n_jobs=-1, verbose=1)(
    joblib.delayed(read_blood_lens)(filename) for filename in blood_filenames
)

blood_NCO_lens = pl.concat([x[0] for x in blood_parts])["CO_active_interval_crossover_prob"] * 1e2
blood_all_lens = pl.concat([x[1] for x in blood_parts])["CO_active_interval_crossover_prob"] * 1e2

fig, ax = plt.subplots(figsize=(6, 3))

sns.histplot(
    [
        pd.Series(np.log10(blood_NCO_lens), name="NCO"),
        pd.Series(np.log10(blood_all_lens), name="All reads"),
    ],
    bins=np.linspace(-5.5, 0.2, 50),
    stat="proportion",
    common_norm=False,
    element="step",
    palette=[NCO_color, background_color],
    fill=False,
    alpha=0.9,
    ax=ax,
    lw=1,
)

ax.set_xlabel("Genetic length (cM)");
ax.set_xticks(
    np.arange(-5, 1),
    [f"$10^{{{x}}}$" for x in np.arange(-5, 1)],
);
ax.set_yticks(
    np.arange(0, 0.08, 0.01),
    [f"{x}" for x in np.arange(0, 0.08, 0.01)],
);
ax.set_ylim(0, 0.075);
ax.legend(
    ax.get_legend().legend_handles,
    ["NCO reads", "All reads"],
    loc="upper left",
);
ax.spines[['right', 'top']].set_visible(False)

fig.savefig(figure_dir / "blood_nco_genetic_lengths.pdf")

In [ ]:
CO_df = CO_NCO_df.filter(pl.col("high_quality_classification_class") == "CO")
NCO_df = CO_NCO_df.filter(pl.col("high_quality_classification_class") == "GC")
sperm_events_df = accepted_events_df.filter(pl.col("dataset").is_in(["TwinsUK", "SL"]))

nco_converted_marker_rate = np.nanmean(NCO_df["grch37_first_converted_marker_poses_rate"]) * 1e6
co_whole_read_rate = sperm_events_df.filter(pl.col("event_type") == "CO")["read_recomb_rate_in_cm_bp"].mean()
nco_whole_read_rate = sperm_events_df.filter((pl.col("dataset") == "TwinsUK") & (pl.col("event_type") == "NCO"))["read_recomb_rate_in_cm_bp"].mean()
complex_whole_read_rate = (
    complex_filtered_df
    .select((pl.col("full_read_crossover_prob") * 1e2 / (pl.col("read_length") * 1e-6)).alias("rate"))
    ["rate"]
    .mean()
)

ks_co_nco = scipy.stats.ks_2samp(CO_lens, NCO_lens).pvalue
ks_co_all = scipy.stats.ks_2samp(CO_lens, all_lens).pvalue
ks_nco_all = scipy.stats.ks_2samp(NCO_lens, all_lens).pvalue

rate_summary = {
    "NCO converted marker cM/Mb": nco_converted_marker_rate,
    "CO whole read cM/Mb": co_whole_read_rate,
    "NCO whole read cM/Mb": nco_whole_read_rate,
    "background whole read cM/Mb": background_whole_read_rate,
    "complex whole read cM/Mb": complex_whole_read_rate,
    "KS CO vs NCO": ks_co_nco,
    "KS CO vs all": ks_co_all,
    "KS NCO vs all": ks_nco_all,
}

rate_summary

## CO positions

In [ ]:
def show_co_map(sample_ids, show_ks=True):
    fig, axs = plt.subplots(4, 6, figsize=(14, 10))

    kses = []
    for ax, chrom in zip(axs.ravel()[:len(aut_chrom_names)], tqdm.tqdm(aut_chrom_names)):
        df = (CO_NCO_df
            .filter(pl.col("sample_id").is_in(sample_ids))
            .filter(pl.col("high_quality_classification_class") == "CO")
            .filter(pl.col("chrom") == chrom)
            .sort("grch37_reference_start_cM")
            .drop_nulls("grch37_reference_start_cM")
        )

        midpoints_in_cms = np.sort((df["grch37_reference_start_cM"] + df["grch37_reference_end_cM"])/2)
        genetic_length_in_cm = rate_maps[chrom].get_cumulative_mass(grch37_chromosome_sizes_in_bp[chrom]-1)*1e2

        ks_pvalue = scipy.stats.ks_1samp(midpoints_in_cms, scipy.stats.uniform(0, genetic_length_in_cm).cdf).pvalue
        kses.append(ks_pvalue)

        ax.plot(midpoints_in_cms, '.')
        ax.plot(
            [0, len(midpoints_in_cms)],
            [0, genetic_length_in_cm],
            color="black",
            ls="--"
        )
        if show_ks:
            ax.set_title(f"{chrom}, KS: p={ks_pvalue:1.2f}");
        else:
            ax.set_title(f"{chrom}");

    combined = scipy.stats.combine_pvalues(kses).pvalue
    fig.supxlabel("Event # (in order along chromosome)");
    fig.supylabel("Position in cM");
    if show_ks:
        fig.suptitle(f"Crossovers, combined p={combined:1.2f}");

    plt.tight_layout()
    return combined

In [ ]:
co_position_combined_p = show_co_map(sample_ids, show_ks=False)

plt.rcParams["pdf.use14corefonts"] = True
fig = plt.gcf()
fig.savefig(figure_dir / "co_positions_genetic_coordinates.pdf")

## Fragile sites

In [28]:
fragile_df = pl.concat([
    pl.read_csv(
        filename,
        new_columns = ["chrom", "start_pos", "end_pos", "fragile_site_name", "something", "something_else"],
        separator = "\t",
        infer_schema = False,
    )
    for filename in glob.glob("/lustre/scratch122/tol/projects/sperm/data/references/10.HumCFS/fragile_site_bed/chr*_fragile_site.bed")
])

In [29]:
dfs = []
for chrom in tqdm.tqdm(aut_chrom_names):
    dfs.append(CO_NCO_df
        .filter(pl.col("chrom") == chrom)
        .select("read_name", "chrom",  "grch38_reference_start", "grch38_reference_end")
        .sort("grch38_reference_start")
        .set_sorted("grch38_reference_start")
        .join_asof(
            (fragile_df
                .filter(pl.col("chrom") == chrom)
                .select(
                    region_start_pos=pl.col("start_pos").cast(int),
                    region_end_pos=pl.col("end_pos").cast(int),
                )
                .sort("region_end_pos")
            ),
            left_on="grch38_reference_start",
            right_on="region_end_pos",
            strategy="forward",
        )
        .with_columns(
            in_fragile_site = (
                pl.col("grch38_reference_start").is_not_null() &
                pl.col("grch38_reference_end").is_not_null() &
                pl.col("region_start_pos").is_not_null() &
                pl.col("region_end_pos").is_not_null() &
                (pl.col("grch38_reference_start") >= pl.col("region_start_pos")) &
                (pl.col("grch38_reference_end") <= pl.col("region_end_pos"))
            )
        )
    )

reads_in_fragile_df = pl.concat(dfs)

CO_NCO_with_fragile_df = (CO_NCO_df
    .join(
        reads_in_fragile_df.select("read_name", "in_fragile_site"),
        on="read_name",
        how="left",
    )
)

xdf = (CO_NCO_with_fragile_df
    .filter(pl.col("high_quality_classification_class") == "GC")
    .select("in_fragile_site", "CO_active_interval_crossover_prob")
)

tbl = (xdf
    .select(
        "in_fragile_site",
        is_low = (pl.col("CO_active_interval_crossover_prob").log(base=10)+2) < -3,
    )
    .group_by("is_low", "in_fragile_site")
    .len()
    .sort("is_low", "in_fragile_site")
)

display(tbl)

fragile_low_high_fisher = scipy.stats.fisher_exact(tbl["len"].to_numpy().reshape((2,2))).pvalue
fragile_high = tbl.filter(~pl.col("is_low")).filter("in_fragile_site")["len"].item() / tbl.filter(~pl.col("is_low"))["len"].sum()
fragile_low = tbl.filter(pl.col("is_low")).filter("in_fragile_site")["len"].item() / tbl.filter(pl.col("is_low"))["len"].sum()
low_fragile = tbl.filter(pl.col("is_low")).filter("in_fragile_site")["len"].item()
low_not_fragile = tbl.filter(pl.col("is_low")).filter(~pl.col("in_fragile_site"))["len"].item()

CO_fragile_df = (CO_NCO_with_fragile_df
    .filter(pl.col("high_quality_classification_class") == "CO")
    .select("in_fragile_site", "CO_active_interval_crossover_prob")
)
co_fragile = CO_fragile_df["in_fragile_site"].mean()
co_fragile_p = scipy.stats.fisher_exact([
    [low_fragile, low_not_fragile],
    [int(CO_fragile_df["in_fragile_site"].sum()), len(CO_fragile_df)],
]).pvalue

recomb_fragile = reads_in_fragile_df["in_fragile_site"].mean()
recomb_fragile_p = scipy.stats.binomtest(
    low_fragile,
    low_fragile + low_not_fragile,
    0.21,
).pvalue

100%|██████████| 22/22 [00:00<00:00, 258.23it/s]


is_low,in_fragile_site,len
bool,bool,u32
false,false,1138
false,true,308
true,false,514
true,true,183


In [30]:
%%time
def read_all_reads_fragile_counts(filename):
    df = (
        pl.scan_parquet(filename)
        .select(
            "chrom",
            "grch38_reference_start",
            "grch38_reference_end",
            "is_high_quality_read",
            "is_contamination",
        )
        .filter("is_high_quality_read")
        .filter(~pl.col("is_contamination"))
        .collect(streaming=True)
    )
    if df.height == 0:
        return pl.DataFrame({"n": [0], "in_fragile": [0]})

    chrom = df["chrom"][0]
    fdf = (df
        .sort("grch38_reference_start")
        .set_sorted("grch38_reference_start")
        .join_asof(
            (fragile_df
                .filter(pl.col("chrom") == chrom)
                .select(
                    region_start_pos=pl.col("start_pos").cast(int),
                    region_end_pos=pl.col("end_pos").cast(int),
                )
                .sort("region_end_pos")
            ),
            left_on="grch38_reference_start",
            right_on="region_end_pos",
            strategy="forward",
        )
        .with_columns(
            in_fragile_site = (
                pl.col("grch38_reference_start").is_not_null() &
                pl.col("grch38_reference_end").is_not_null() &
                pl.col("region_start_pos").is_not_null() &
                pl.col("region_end_pos").is_not_null() &
                (pl.col("grch38_reference_start") >= pl.col("region_start_pos")) &
                (pl.col("grch38_reference_end") <= pl.col("region_end_pos"))
            )
        )
    )
    return fdf.select(
        n=pl.len(),
        in_fragile=pl.col("in_fragile_site").sum(),
    )

all_reads_fragile_counts_df = pl.concat(
    joblib.Parallel(n_jobs=4, verbose=1)(
        joblib.delayed(read_all_reads_fragile_counts)(filename) for filename in reads_filenames
    )
)

all_reads_fragile_n = all_reads_fragile_counts_df["n"].sum()
all_reads_fragile_k = all_reads_fragile_counts_df["in_fragile"].sum()
all_reads_fragile = all_reads_fragile_k / all_reads_fragile_n
all_reads_fragile_p = scipy.stats.binomtest(low_fragile, low_fragile + low_not_fragile, 0.233).pvalue

blood_nco_fragile = 270/(270+765)

fragile_summary = {
    "sperm low-rate NCO fragile overlap": fragile_low,
    "sperm high-rate NCO fragile overlap": fragile_high,
    "sperm low/high NCO Fisher P": fragile_low_high_fisher,
    "CO fragile overlap": co_fragile,
    "CO fragile Fisher P": co_fragile_p,
    "all recombinant fragile overlap": recomb_fragile,
    "all recombinant binomial P": recomb_fragile_p,
    "all reads fragile overlap": all_reads_fragile,
    "all reads binomial P": all_reads_fragile_p,
    "blood NCO fragile overlap": blood_nco_fragile,
}

fragile_summary

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.3s
/nfs/users/nfs_r/rs42/.local/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


CPU times: user 542 ms, sys: 107 ms, total: 648 ms
Wall time: 8.46 s


[Parallel(n_jobs=4)]: Done 330 out of 330 | elapsed:    8.5s finished


{'sperm low-rate NCO fragile overlap': 0.26255380200860834,
 'sperm high-rate NCO fragile overlap': 0.21300138312586445,
 'sperm low/high NCO Fisher P': np.float64(0.011600534325133959),
 'CO fragile overlap': 0.21586037966932026,
 'CO fragile Fisher P': np.float64(1.0312967080661038e-07),
 'all recombinant fragile overlap': 0.21913544668587898,
 'all recombinant binomial P': np.float64(0.0009420719877058272),
 'all reads fragile overlap': 0.23120508104107446,
 'all reads binomial P': np.float64(0.06628692089988888),
 'blood NCO fragile overlap': 0.2608695652173913}

## Summary

In [31]:
key_values = {
    **rate_summary,
    "CO position combined P": co_position_combined_p,
    **fragile_summary,
}

key_values

{'NCO converted marker cM/Mb': np.float64(14.372525507713261),
 'CO whole read cM/Mb': 8.844190013848616,
 'NCO whole read cM/Mb': 4.212605150077757,
 'background whole read cM/Mb': 1.1086666108085381,
 'complex whole read cM/Mb': 2.4091740182899,
 'KS CO vs NCO': np.float64(4.2028392360085436e-101),
 'KS CO vs all': np.float64(0.0),
 'KS NCO vs all': np.float64(4.8750174689987e-134),
 'CO position combined P': np.float64(1.2813330789318017e-06),
 'sperm low-rate NCO fragile overlap': 0.26255380200860834,
 'sperm high-rate NCO fragile overlap': 0.21300138312586445,
 'sperm low/high NCO Fisher P': np.float64(0.011600534325133959),
 'CO fragile overlap': 0.21586037966932026,
 'CO fragile Fisher P': np.float64(1.0312967080661038e-07),
 'all recombinant fragile overlap': 0.21913544668587898,
 'all recombinant binomial P': np.float64(0.0009420719877058272),
 'all reads fragile overlap': 0.23120508104107446,
 'all reads binomial P': np.float64(0.06628692089988888),
 'blood NCO fragile overla